In [1]:
import sys
import json
import yaml
import os
import pandas as pd
from pathlib import Path
REPO_ROOT = Path.home() / "destriping-GLM-rebuttals"
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from src.spatialAdata.loading import load_spatialAdata

bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


In [2]:
def get_nuclei_stats(path_data: Path, cell_id_label: str) -> dict:
    """Load sdata and return n (nuclei bins) and n_nuclei (unique nuclei)."""
    data = load_spatialAdata(path_data)
    col = data.adata.obs[cell_id_label]
    return {
        "n": int(col.notna().sum()),
        "n_nuclei": int(col.dropna().nunique()),
    }

In [3]:
runs_dict = yaml.safe_load(
    open("experiments/benchmark_output_files/time_requirements/glum_runs.yaml")
)
rows = []

for dataset, path in runs_dict.items():
    path = Path(path)

    # Find time_dict.json (depth may vary across datasets)
    time_json = next(path.rglob("time_dict.json"))
    with open(time_json) as f:
        time_data = json.load(f)

    hydra_cfg_run_path = next((path).rglob("config.yaml"))
    with open(hydra_cfg_run_path) as f:
        hydra_cfg_run = yaml.safe_load(f)

    path_data = hydra_cfg_run["dataset"]["path_data"]
    cell_id_label = hydra_cfg_run["dataset"]["cell_id_label"]
    parallel = hydra_cfg_run["model_args"].get("parallel", "not found")
    nuclei_stats = get_nuclei_stats(path_data, cell_id_label)

    rows.append({
        "dataset": dataset,
        **nuclei_stats,
        "fitting_time (min)": round(time_data["fitting_time"] / 60, 2),
        "parallel": parallel
    })

df = pd.DataFrame(rows).sort_values("n_nuclei").reset_index(drop=True)
df

,dataset,n,n_nuclei,fitting_time (min),parallel
0,mouse_brain,853259,61842,39.45,False
1,zebrafish_head,344157,121779,108.15,False
2,human_lymph_node,1383947,292813,253.50,False
3,mouse_embryo,2638892,448092,222.55,False


In [4]:
out_path = REPO_ROOT / "results/my_notebooks/computational_requirements/time.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print(f"Saved to {out_path.resolve()}")

Saved to /nfs/storage-raetsch.inf.ethz.ch/mnt/raetsch/euler/home/pmalsot/bin2cell/results/my_notebooks/computational_requirements/time.csv


## Getting the destripe time from the memory benchmark

In [5]:
runs_dict = yaml.safe_load(
    open("experiments/benchmark_output_files/memory_requirements/destriping.yaml")
)
rows = []

for dataset, path in runs_dict.items():
    path = Path(path)

    # Find time_dict.json (depth may vary across datasets)
    time_json = next(path.rglob("time_dict.json"))
    with open(time_json) as f:
        time_data = json.load(f)

    hydra_cfg_run_path = next((path).rglob("config.yaml"))
    with open(hydra_cfg_run_path) as f:
        hydra_cfg_run = yaml.safe_load(f)

    path_data = hydra_cfg_run["dataset"]["path_data"]
    cell_id_label = hydra_cfg_run["dataset"]["cell_id_label"]
    parallel = hydra_cfg_run.get("model_args", {}).get("parallel", "not found")
    nuclei_stats = get_nuclei_stats(path_data, cell_id_label)

    rows.append(
        {
            "dataset": dataset,
            **nuclei_stats,
            "destriping_time (min)": round(time_data["destriping_time"] / 60, 2),
            "parallel": parallel,
        }
    )

df = pd.DataFrame(rows).sort_values("n_nuclei").reset_index(drop=True)
df

,dataset,n,n_nuclei,destriping_time (min),parallel
0,mouse_brain,853259,61842,0.53,not found
1,zebrafish_head,344157,121779,0.15,not found
2,human_lymph_node,1383947,292813,0.25,not found
3,mouse_embryo,2638892,448092,0.77,not found


In [6]:
out_path = REPO_ROOT / "results/my_notebooks/computational_requirements/time_destriping.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print(f"Saved to {out_path.resolve()}")

Saved to /nfs/storage-raetsch.inf.ethz.ch/mnt/raetsch/euler/home/pmalsot/bin2cell/results/my_notebooks/computational_requirements/time_destriping.csv
